# Phase 2B — Percakapan OpenAI + function calling
Notebook ini menguji percakapan routing yang benar-benar melewati OpenAI API. Preference tidak di-hardcode ke hasil AI: setiap turn mengirim pesan natural-language, lalu preference dari respons turn sebelumnya diteruskan sebagai `conversation_preferences`.

Urutan: (1) tanpa preference awal, (2) memakai preference state sebelumnya, (3) membangun preference dari bahasa natural, (4) respons berikutnya menyesuaikan preference. Semua function call dan hasil routing dicetak jelas. Isi `OPENAI_API_KEY` dan `OPENAI_MODEL` di `backend/.env`. Notebook ini sengaja live-only.

In [1]:
from pathlib import Path
import sys, json, tempfile
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'backend' / 'jakroute').is_dir()), None)
assert ROOT, 'Buka notebook dari folder paket yang sudah diekstrak lengkap.'
sys.path.insert(0, str(ROOT / 'backend'))
DATA = ROOT / 'backend' / 'data'
from jakroute.providers import load_json
from jakroute.config import Settings

from dataclasses import replace
from jakroute.service import RouteService
RUN_LIVE_OPENAI=True
work=tempfile.TemporaryDirectory(prefix='jakroute_agent_conversation_')
settings=replace(Settings.from_env(),db_path=str(Path(work.name)/'forum.sqlite3'),agent_mode='openai',mapid_mode='demo',weather_mode='demo',forum_mode='demo')
assert RUN_LIVE_OPENAI
assert settings.openai_api_key and settings.openai_model, 'Isi OPENAI_API_KEY dan OPENAI_MODEL di backend/.env dahulu.'
service=RouteService(settings)
print('OPENAI LIVE — preference state berasal dari respons turn sebelumnya')

OPENAI LIVE — preference state berasal dari respons turn sebelumnya


In [2]:
def show_response(turn_name,user_message,request,response):
    print('\n'+'='*78)
    print(f'TURN {turn_name}')
    print(f'USER: {user_message}')
    print('REQUEST PREFERENCE STATE:')
    print(json.dumps(request.get('conversation_preferences',{}),ensure_ascii=False,indent=2))
    print('AI INTENT:')
    print(json.dumps(response.get('intent',{}),ensure_ascii=False,indent=2))
    print('FUNCTION CALLS:')
    for trace in response.get('tool_trace',[]):
        print(f"- {trace['name']} | {json.dumps(trace['arguments'],ensure_ascii=False)} | {trace['source']} | {trace['status']}")
    print('ROUTES:')
    for route in response.get('routes',[]):
        print(f"- {route.get('label')}: status={route.get('status')}, walking_m={route.get('walking_m')}, duration_s={route.get('duration_s')}, connectors={route.get('connectors_used')}")
    print('SELECTED:',response.get('selected_route_id'))
    print('PREFERENCES AFTER TURN:')
    print(json.dumps(response.get('preferences',{}),ensure_ascii=False,indent=2))
    return response

def run_turn(turn_name,user_message,previous_response=None):
    request={'origin_id':'entrance_west','message':user_message}
    if previous_response:
        previous_preferences=previous_response.get('preferences',{})
        previous_intent=previous_response.get('intent',{})
        # These are copied from the previous AI response, never hardcoded.
        # Sending them as explicit state also keeps this notebook compatible
        # with an older backend that does not yet read conversation_context.
        request['preferences']=previous_preferences
        request['conversation_preferences']=previous_preferences
        request['conversation_context']={'intent':previous_intent}
        for key in ('origin_id','destination_id'):
            if previous_intent.get(key): request[key]=previous_intent[key]
    response=service.recommend(request)
    assert response['status']=='ok', response
    return show_response(turn_name,user_message,request,response)

responses=[]
responses.append(run_turn('1 — tanpa preference awal','Saya mau ke peron 1.'))
responses.append(run_turn('2 — memakai preference state sebelumnya','Tampilkan lagi rute menuju tujuan yang sama.',responses[-1]))
responses.append(run_turn('3 — membangun preference dari bahasa natural','Saya tidak bisa naik tangga. Gunakan akses yang bebas anak tangga.',responses[-1]))
responses.append(run_turn('4 — respons berikutnya menyesuaikan preference','Sekarang arahkan saya ke peron 1.',responses[-1]))


TURN 1 — tanpa preference awal
USER: Saya mau ke peron 1.
REQUEST PREFERENCE STATE:
{}
AI INTENT:
{
  "origin_id": "entrance_west",
  "destination_id": "platform_1",
  "via_indoor_ids": [],
  "focus_mode": "best_fit",
  "preferences": {
    "avoid_stairs": false,
    "step_free": false,
    "preferred_access": "any",
    "time_priority": 1.0,
    "walking_priority": 1.0,
    "crowd_priority": 1.0,
    "max_walk_m": null,
    "required_facilities": []
  },
  "clarification": null
}
FUNCTION CALLS:
- route_indoor_personalized | {"origin_id": "entrance_west", "destination_id": "platform_1", "mode": "best_fit"} | openai_function_call | ok
- route_indoor_personalized | {"origin_id": "entrance_west", "destination_id": "platform_1", "mode": "fastest"} | openai_function_call | ok
- route_indoor_plain | {"mode": "min_walk", "origin_id": "entrance_west", "destination_id": "platform_1"} | openai_function_call | ok
ROUTES:
- Rute Paling Sesuai: status=ok, walking_m=76.63, duration_s=78.9, connect

In [3]:
initial=responses[0]['preferences']
retained=responses[1]['preferences']
built=responses[2]['preferences']
adapted=responses[3]
assert sum(initial[key] for key in ('time_priority','walking_priority','crowd_priority'))>0
assert built['step_free'] or built['avoid_stairs'], built
assert all(route.get('connectors_used')==['elevator_link'] for route in adapted['routes'] if route.get('status')=='ok')
print('\nPASS: preference tidak wajib di awal, state dipertahankan, preference dibangun dari bahasa natural, dan turn berikutnya memakai lift.')


PASS: preference tidak wajib di awal, state dipertahankan, preference dibangun dari bahasa natural, dan turn berikutnya memakai lift.


## Kontrak endpoint untuk Flutter
Flutter menyimpan `preferences` dari respons lalu mengirimkannya kembali sebagai `conversation_preferences` pada pesan berikutnya.

In [4]:
from fastapi.testclient import TestClient
from jakroute.api import create_app
with TestClient(create_app(settings,service)) as client:
    response=client.post('/recommend-route',json={'origin_id':'entrance_west','message':'Saya tidak bisa naik tangga.','conversation_preferences':responses[1]['preferences']})
    assert response.status_code==200,response.text
    payload=response.json()
    print('ENDPOINT RESPONSE KEYS:',list(payload))
    print('ENDPOINT STATUS:',payload['status'])
work.cleanup()

c:\Users\owen\anaconda3\envs\torch\lib\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
  from starlette.testclient import TestClient as TestClient  # noqa


ENDPOINT RESPONSE KEYS: ['status', 'question', 'routes', 'intent', 'agent_mode']
ENDPOINT STATUS: clarification_required
